# 05-05 RunnableParallel: 여러 체인을 동시에 실행하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnableParallel`로 여러 체인을 동시에 실행하고 결과를 딕셔너리로 합치는 방법을 구현할 수 있어요
- 딕셔너리 형태의 자동 래핑 원리를 이해하고, 세 가지 동등한 표기법을 설명할 수 있어요
- `itemgetter`를 활용하여 동일한 입력 딕셔너리에서 서로 다른 키를 추출하는 패턴을 적용할 수 있어요
- 순차 실행과 병렬 실행의 처리 시간 차이를 측정하고 설명할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `RunnablePassthrough`와 LCEL 파이프 연산자(`|`) 사용법
- `operator.itemgetter`의 기본 사용법
- 파이썬의 멀티스레딩(Multi-threading) 개념 (심화 이해에 도움)

---

`RunnableParallel`은 여러 `Runnable`을 병렬로 실행하고, 그 결과를 딕셔너리로 반환해요.

아래 다이어그램은 병렬 실행의 흐름을 보여줘요:

```mermaid
flowchart TD
    IN["입력 딕셔너리"]:::input
    FORK["RunnableParallel<br/>분기 시작"]:::process
    C1["capital_chain<br/>수도 조회"]:::process
    C2["area_chain<br/>면적 조회"]:::process
    JOIN["결과 병합<br/>{capital: ..., area: ...}"]:::process
    OUT["최종 출력 딕셔너리"]:::output

    IN --> FORK
    FORK --> C1
    FORK --> C2
    C1 --> JOIN
    C2 --> JOIN
    JOIN --> OUT

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**주요 특징:**

- 여러 작업을 동시에 실행하여 성능 향상
- 각 Runnable의 결과를 키로 구분하여 반환
- 딕셔너리 형태로 결과를 통합

In [2]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [3]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

## 1. RunnableParallel 기본 사용법

`RunnableParallel`은 딕셔너리 형태로 여러 Runnable을 병렬로 실행해요.

**참고:** 딕셔너리 형태로 작성하면 자동으로 `RunnableParallel`로 래핑돼요.

다음 세 가지 방식은 모두 동일하게 처리돼요:

```python
# 방법 1: 딕셔너리 자동 래핑 (가장 간결)
{"capital": capital_chain, "area": area_chain}

# 방법 2: 딕셔너리를 명시적으로 전달
RunnableParallel({"capital": capital_chain, "area": area_chain})

# 방법 3: 키워드 인자 사용
RunnableParallel(capital=capital_chain, area=area_chain)
```

> **실무 팁:** LCEL 파이프라인에서 `{...}` 형태의 딕셔너리를 만나면 LangChain은 내부적으로 `RunnableParallel`로 처리해요. 별도로 명시하지 않아도 병렬 실행이 적용돼요.

In [4]:
# ============================================================
# TODO: RunnableParallel로 두 체인을 병렬 실행하는 코드를 완성하세요
# 힌트: RunnableParallel(capital=capital_chain, area=area_chain)
# 예상 결과: {'capital': '대한민국의 수도는 서울입니다.', 'area': '약 100,210 km²...'}
# ============================================================

# ---------------------------------------------------
# RunnableParallel 기본 예제 — 병렬로 두 체인 실행하기
# ---------------------------------------------------
capital_chain = (
  ChatPromptTemplate.from_template("{country}의 수도는 어디입니까?")
  | model
  | StrOutputParser()
)

area_chain = (
  ChatPromptTemplate.from_template("{country}의 면적은 얼마입니까?")
  | model
  | StrOutputParser()
)

parallel_chain = RunnableParallel(
  capital=capital_chain,
  area=area_chain
)

res = parallel_chain.invoke({"country": "대한민국"})
print(res)


{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 면적은 약 100,210 평방킬로미터(약 38,691 평방마일)입니다. 이 면적에는 한반도의 남쪽 부분인 대한민국과 그에 속하는 여러 섬들이 포함됩니다.'}


## 2. itemgetter를 활용한 RAG 예제

`operator.itemgetter`는 딕셔너리 입력에서 특정 키의 값을 추출하는 도구예요.

`RunnableParallel` 안에서 입력 딕셔너리의 서로 다른 키를 각각의 체인에 전달할 때 자주 사용해요.

In [5]:
# ============================================================
# TODO: itemgetter를 활용하여 입력 딕셔너리에서 값을 추출하는 RAG 체인을 완성하세요
# 힌트: itemgetter("question") — 입력 딕셔너리의 "question" 키 값 추출
# 예상 결과: "파이썬은 데이터 과학과 머신러닝 분야에서 널리 사용됩니다."
# ============================================================

# ---------------------------------------------------
# itemgetter + RunnableParallel — RAG 패턴 구현
# ---------------------------------------------------

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# 1단계: 벡터 저장소 생성 (간단한 예시용 데이터)
vectorstore = Chroma.from_texts(
    ["파이썬은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다."],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

# 문서 리스트를 하나의 문자열로 합치는 헬퍼 함수
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

# RAG 프롬프트 템플릿
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
{language}로 답변해주세요.

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 2단계: itemgetter로 입력에서 필요한 값 추출
# - itemgetter("key"): dict["key"] 와 동일하지만 Runnable로 사용 가능
# - context: question으로 벡터 검색 후 포맷팅
# - question, language: 입력 딕셔너리에서 직접 추출

from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "language": itemgetter("language")
    }
    | prompt
    | model
    | StrOutputParser()
)

res = rag_chain.invoke({
    "question": "파이썬은 어떤 분야에서 사용되나요?",
    "language": "한국어"
})

res


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 4 is greater than number of elements in index 1, updating n_results = 1


'파이썬은 데이터 과학과 머신러닝을 비롯하여 웹 개발, 자동화 스크립팅, 데이터 분석, 인공지능, 과학 계산, 게임 개발 등 다양한 분야에서 사용됩니다.'

## 3. 서로 다른 입력 변수를 사용하는 예제

`RunnableParallel` 안의 각 체인이 서로 다른 입력 변수를 사용할 수 있어요. 이 경우 `invoke()` 호출 시 모든 체인이 필요로 하는 키를 모두 포함한 딕셔너리를 전달해야 해요.

In [ ]:
# ============================================================
# TODO: 서로 다른 입력 변수(country1, country2)를 사용하는 병렬 체인을 완성하세요
# 힌트: 각 체인이 다른 키를 사용하더라도 invoke()에 모든 키를 포함하면 됨
# 예상 결과: {'capital': '서울', 'area': '미국의 면적은 약 9,830,000 km²'}
# ============================================================

# ---------------------------------------------------
# 서로 다른 입력 변수를 사용하는 병렬 체인
# ---------------------------------------------------
capital_chain_1 = (
    ChatPromptTemplate.from_template("{country1}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain_2 = (
    ChatPromptTemplate.from_template("{country2}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)


parallel_chain = RunnableParallel(
  capital = capital_chain_1,
  area = area_chain_2
)

res = parallel_chain.invoke({
  "country1": "대한민국",
  "country2": "천조국"
})

res

{'capital': '대한민국의 수도는 서울입니다.',
 'area': '"천조국"이라는 표현은 일반적으로 미국을 가리키는 용어로 사용됩니다. 미국의 면적은 약 9,834,000 제곱킬로미터(3,796,742 제곱마일)로, 이는 세계에서 세 번째로 큰 나라입니다. 참여하고 있는 지역이나 특정 주를 포함할 수도 있지만, 전체 미국의 면적 기준입니다.'}

## 4. 병렬 실행 성능 비교

`RunnableParallel`을 사용하면 여러 작업을 병렬로 실행하여 전체 실행 시간을 단축할 수 있어요.

순차 실행과 병렬 실행의 시간 차이를 비교해볼게요.

> **실무 팁:** `RunnableParallel`의 성능 이점은 각 체인이 독립적이고 I/O 대기 시간(API 호출 등)이 있을 때 특히 커요. 체인 수가 많아질수록 순차 실행 대비 절감 효과가 커져요. 반면 체인들이 서로 의존 관계에 있다면 순차 실행을 사용해야 해요.

In [ ]:
import time

# ============================================================
# TODO: sequential_execution()과 parallel_execution() 함수를 구현하고 성능을 비교하세요
# 힌트: time.time()으로 실행 전후 시간을 측정
# 예상 결과: 병렬 실행이 순차 실행보다 빠름 (1.x배 향상)
# ============================================================

# ---------------------------------------------------
# 성능 비교: 순차 실행 vs 병렬 실행
# ---------------------------------------------------


## 마무리 요약

### RunnableParallel의 핵심 개념

| 개념 | 설명 |
|------|------|
| 자동 래핑 | LCEL에서 `{...}` 딕셔너리는 자동으로 `RunnableParallel`로 변환 |
| 결과 형태 | 각 체인의 결과를 키-값 쌍으로 묶어 딕셔너리 반환 |
| 독립성 요건 | 병렬 체인들은 서로 의존 관계가 없어야 함 |
| 성능 이점 | API 호출 등 I/O 대기가 있을 때 순차 실행 대비 속도 향상 |

### 핵심 요점

- `RunnableParallel`은 독립적인 여러 체인을 동시에 실행하여 전체 소요 시간을 줄여줘요
- 딕셔너리 형태 `{key: runnable, ...}`로 작성하면 자동으로 병렬 실행이 적용돼요
- `itemgetter`로 입력 딕셔너리에서 필요한 키만 추출해 각 체인에 전달할 수 있어요
- 서로 다른 입력 변수를 사용하는 체인도 하나의 `RunnableParallel`에 묶을 수 있어요

### 다음 노트북 예고

다음 노트북(`06-Configure.ipynb`)에서는 체인을 실행할 때 런타임(Runtime)에 동적으로 설정을 변경하는 방법을 배워요. 모델이나 프롬프트를 호출 시점에 교체하는 `configurable_fields`와 `configurable_alternatives`를 살펴볼게요.